# 02 · Can spontaneous fluctuations predict a response?

**The Stochastic Fly · Research assignment**

In Notebook 01 you built a random-sequential network from directed synaptic contact counts, then checked its stationary currents and cycle affinities. The symmetric chains agreed with the Gibbs law and had currents at numerical precision. With the directed weights, the maximum current was about $1.31\times10^{-2}$ in the synthetic example and $6.12\times10^{-5}$ in the three-cell MaleCNS example. The latter retained only a small fraction of the cells' incoming effective strength.

Here you will use those **same networks** to answer a different question. Can a time series at rest predict the response to a weak step in the external field? You will calculate both sides of an equilibrium identity by exact finite-state propagation. This removes sampling error from the comparison and gives us a reference for the trajectory-based estimator we will need on larger networks. The directed model tests how much predictive accuracy is lost for particular choices of input and readout. Stationary currents alone do not determine that error.

This assignment assumes Notebook 01 is completed. It provides a setup cell that reuses that notebook's code, data, and constants. The remaining code cells are yours. Study each section, implement the stated functions, and write your interpretation at the end.

## Provided setup · Reuse the first assignment

Run this cell from the same Python environment as Notebook 01. It runs the saved Python cells of your first assignment in a separate namespace, including the MaleCNS loader and the numerical checks. Their display output is captured to keep this notebook readable. The source notebook and its saved outputs are read from disk. If you keep your completed assignment under a different name, specify it in `NOTEBOOK_01_OVERRIDE`.

The setups share $\beta$, $g$, $\theta$, and the baseline field $h^{(0)}$. Each row of `states` stores a binary configuration $\sigma\in\{0,1\}^{N}$. The notation $\sigma(t)$ denotes the trajectory's configuration at time $t$, and $\sigma_i(t)$ denotes its $i$th component. The corresponding $\{-1,+1\}$ spin is $s_i=2\sigma_i-1$. The field $h_i$ is added to the local field $\phi_i$. The gain $g$ multiplies only the recurrent term.

The two fixtures define **cell indices**, with zero-based indexing into the saved order. In the MaleCNS example the two LPLC2 cells form the forward input set and DNp01 is the readout. The association motivates the choice of populations for this exercise. The three-cell dynamics uses only the selected cells.

In [ ]:
from pathlib import Path
import json

from IPython.display import display
from IPython.utils.capture import capture_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_01_OVERRIDE = None  # Set a Path here if your completed copy was renamed.
NOTEBOOK_01_NAME = "01_from_malecns_to_detailed_balance.ipynb"
cwd = Path.cwd()

if NOTEBOOK_01_OVERRIDE is not None:
    notebook_01 = Path(NOTEBOOK_01_OVERRIDE).expanduser().resolve()
else:
    notebook_01 = next(
        (candidate for folder in (cwd, *cwd.parents)
         for candidate in (folder / NOTEBOOK_01_NAME,
                           folder / "notebooks" / "assignments" / NOTEBOOK_01_NAME)
         if candidate.is_file()),
        None,
    )
if notebook_01 is None or not notebook_01.is_file():
    raise FileNotFoundError("Set NOTEBOOK_01_OVERRIDE to your completed Notebook 01.")

previous_notebook = json.loads(notebook_01.read_text(encoding="utf-8-sig"))
prior = {}
with capture_output():
    for cell in previous_notebook["cells"]:
        if cell["cell_type"] == "code":
            source = "".join(cell["source"])
            exec(compile(source, f"{notebook_01.name}:{cell.get('id', 'cell')}", "exec"), prior)

required = (
    "transition_matrix", "stationary_distribution", "stationary_current",
    "split_coupling", "J_toy", "J_malecns", "states_toy", "states_malecns",
    "BETA", "GAIN", "THRESHOLD", "REST_FIELD",
)
missing = [name for name in required if name not in prior]
if missing:
    raise RuntimeError(f"Complete Notebook 01 before continuing. Missing: {', '.join(missing)}")

transition_matrix = prior["transition_matrix"]
stationary_distribution = prior["stationary_distribution"]
stationary_current = prior["stationary_current"]
split_coupling = prior["split_coupling"]
BETA = prior["BETA"]
GAIN = prior["GAIN"]
THRESHOLD = prior["THRESHOLD"]
REST_FIELD = prior["REST_FIELD"]

# (Coupling, state array, forward input indices, forward readout indices)
FIXTURES = {
    "Synthetic fixture": (prior["J_toy"], prior["states_toy"], (0,), (1,)),
    "MaleCNS": (prior["J_malecns"], prior["states_malecns"], (0, 1), (2,)),
}
ALPHAS = (0.0, 0.25, 0.5, 0.75, 1.0)
TIMES = np.arange(61)  # Time in elementary random-sequential updates.
EPSILON = 0.001
EPSILON_GRID = np.array([0.1, 0.03, 0.01, 0.003, 0.001, 0.0003])
ATOL = 1e-12
FDT_TOL = 1e-7  # Absolute tolerance for the default finite-step check.

print(f"Reused {notebook_01.name}.")
for name, (J, states, input_indices, readout_indices) in FIXTURES.items():
    print(f"{name}: {J.shape[0]} cells, {len(states)} configurations, "
          f"input {input_indices}, readout {readout_indices}.")

## 1 · Define the field step and the observables

Let $I$ be the input population and $R$ the readout population. Use a stimulus mask $b_i=\mathbf 1_{\{i\in I\}}$ and define
$$
B(\sigma)=b^\mathsf{T}\sigma=\sum_{i\in I}\sigma_i,\qquad
O(\sigma)=\frac{1}{|R|}\sum_{i\in R}\sigma_i .
$$
The perturbation adds $\epsilon b_i$ to the resting field of cell $i$, starting just before the first update. The amplitude $\epsilon$ is the increment **per input cell**. This explains why $B$ is a sum and $O$ is an average. At equilibrium, a field increment changes the energy by $-\epsilon B(\sigma)$, so $B$ is the observable conjugate to the field.

**Exercise 1.** Define `make_observables(states, input_indices, readout_indices) -> (b, B, O)`. Return a mask of shape `(n_cells,)` and two vectors of shape `(n_states,)`. Evaluate all three for the forward populations in both fixtures. Check that the all-zero configuration has $B=O=0$ and the all-one configuration has $B=|I|$, $O=1$. Confirm that $B$ changes only when an input cell changes and $O$ only when a readout changes.

**Python starting point.** Multiplying the state array by the input mask gives one sum per row. Indexing the readout columns and taking their row mean gives one observable value per row.

In [ ]:
# Your code for Exercise 1.
# Define the input mask and the two observable vectors.

## 2 · Measure the response by propagating the full distribution

Fix one $\alpha$ and write $P_0$ for its kernel under the resting field. Find the distribution $\pi_0$ such that $\pi_0P_0=\pi_0$. In this notebook, the subscript $0$ on $P_0$, $\pi_0$, or a covariance refers to **zero perturbation at the current $\alpha$**. The reversible condition is denoted separately by $\alpha=0$.

At time zero all trials have the same distribution $\pi_0$. The step changes the transition matrix to $P_\epsilon$. With row-vector probabilities, the mean in the perturbed branch is
$$
m_\epsilon(t)=\pi_0 P_\epsilon^t O .
$$
Here $O$ denotes the column vector of readout values. The first value, $m_\epsilon(0)=\pi_0O$, precedes any update under the field. Starting the perturbed branch in its *own* stationary distribution would give a stationary mean at every time, thereby losing the transient response.

The step susceptibility of interest is the derivative at zero field increment. Estimate it by a centered finite difference,
$$
\chi_{OB}(t)=\left.\frac{\partial m_\epsilon(t)}{\partial\epsilon}\right|_{\epsilon=0},
\qquad
\widehat\chi_{OB,\epsilon}(t)=\frac{m_{+\epsilon}(t)-m_{-\epsilon}(t)}{2\epsilon}.
$$
For a smooth response, the leading truncation error of the centered difference is proportional to $\epsilon^2$. Very small $\epsilon$ can amplify floating-point cancellation.

**Exercise 2.** Implement `propagate_expectation(P, pi_initial, observable, n_steps)` and `step_susceptibility(P_plus, P_minus, pi_0, O, epsilon, n_steps)`. Use repeated vector–matrix multiplication and return an array that includes $t=0$. Both branches start from identical copies of $\pi_0$. Use the signed field perturbations $h^{(0)}\pm\epsilon b$ when constructing the two kernels.

**Checks.** An unperturbed chain started at $\pi_0$ has a constant mean. The susceptibility at $t=0$ is zero. For disjoint input and readout sets, it also vanishes after one elementary update: the field changes only the resampling probability of input cells, and an update of an input cell leaves the instantaneous readout unchanged.

In [ ]:
# Your code for Exercise 2.
# Propagate each field branch and compute its centered difference.

## 3 · Predict the response using only the resting chain

The lagged covariance pairs the input at time zero with the readout at time $t$,
$$
C_{OB}(t)=\operatorname{Cov}_0[O(\sigma(t)),B(\sigma(0))].
$$
Conditional on the starting configuration, the expected future readout is $(P_0^t O)(\sigma(0))$. Centering both observables under $\pi_0$ therefore gives a computational formula,
$$
C_{OB}(t)=\bigl(\pi_0\odot[B-\pi_0B]\bigr)\,\bigl(P_0^t[O-\pi_0 O]\bigr),
$$
where $\odot$ means elementwise multiplication. Use $\pi_0$ to weight the starting configurations. The lagged readout propagates as a **column**, so each update applies `P_0 @ future_readout`. In Exercise 2 you propagated a **row** distribution using `probability @ P`. These operations answer different conditional-expectation questions.

At equilibrium the fluctuation–dissipation relation for a field step is
$$
\chi_{OB}^{\mathrm{eq}}(t)=\beta[C_{OB}(0)-C_{OB}(t)].
$$

**Why the formula works here.** At $\alpha=0$, each $P_\epsilon$ preserves the Gibbs law for the shifted energy $E-\epsilon B$, so $\pi_\epsilon P_\epsilon^t=\pi_\epsilon$. Differentiate this identity at zero. The term from differentiating $P_\epsilon^t$ is $\pi'_0(\mathrm{Id}-P_0^t)$, where $\mathrm{Id}$ is the identity matrix. Differentiating the Gibbs law gives $\pi'_0(\sigma)=\beta\pi_0(\sigma)[B(\sigma)-\pi_0B]$. Multiplying by $O$ gives the difference between the equal-time and lagged covariances. This is the discrete-time calculation for our update rule. For the directed chain we can still compute the covariance and the same formula as a candidate predictor. Its equality with the response is an empirical comparison in this assignment. [Baiesi, Maes & Wynants, *Fluctuations and response of nonequilibrium states*](https://arxiv.org/abs/0902.3955)

**Exercise 3.** Implement `stationary_cross_covariance(P_0, pi_0, B, O, n_steps)` and form the predictor from the returned vector. Check $C_{OB}(0)$ by calculating the weighted covariance directly from the state table. For the reversible chain, check the reciprocal relation $C_{OB}(t)=C_{BO}(t)$ by swapping the *observable vectors* $B$ and $O$. This check uses the same vectors without changing their scaling.

**Independent one-cell check.** Use `J = np.zeros((1, 1))`, `states = np.array([[0], [1]])`, and the single cell as both input and readout. At each update the cell is assigned a new value independently of its previous value. If $p=[1+\exp(-\beta[h^{(0)}-\theta])]^{-1}$, then $C_{OB}(0)=p(1-p)$, $C_{OB}(t)=0$ for $t\ge1$, and the exact susceptibility at $t\ge1$ is $\beta p(1-p)$. Check both paths against these values.

In [ ]:
# Your code for Exercise 3.
# Propagate the centered readout and validate the one-cell example.

## 4 · Validate the equilibrium reference and choose a small field

The first validation is a reproducibility test. At $\alpha=0$, the kernel must have detailed balance and the finite-difference response should approach the covariance prediction as the field increment shrinks. This uses the same normalized $J$ from Notebook 01. Construct $J_\alpha=S+\alpha K$ after that normalization.

**Exercise 4.** For each forward population pair at $\alpha=0$, build the resting kernel, solve for $\pi_0$, and compute the covariance predictor. For every value in `EPSILON_GRID`, compute the step susceptibility and its maximum absolute difference from the predictor over `TIMES`. Show a table or a log–log plot of this error against $\epsilon$ for each fixture.

- Show that the error at the default `EPSILON` falls below `FDT_TOL` in both fixtures. Treat this as a check in units of susceptibility.
- Look for a range in which shrinking $\epsilon$ reduces the error approximately quadratically. Floating-point cancellation can make the smallest errors level off.
- Select an amplitude for the remaining comparison, with one smaller amplitude as a check. Keep the time grid and all other parameters fixed.
- If a discrepancy persists at $\alpha=0$, check the initial distribution, the sign of the perturbation, the sum versus mean convention for $B$, and the covariance lag direction.

**Suggested output.** A tidy table with columns `fixture`, `epsilon`, and `max_abs_error`. Record the amplitude used in the next experiments.

In [ ]:
# Your code for Exercise 4.
# Validate the equilibrium benchmark across field amplitudes.

## 5 · Measure where the equilibrium prediction fails

For every condition, estimate the resting covariance under that condition's own invariant law. This keeps both response and predictor anchored to the same operating point. The equilibrium formula uses rest statistics, so any difference we measure is conditional on the model, field protocol, and readout. A nonzero stationary current establishes irreversibility of the chain. A nonzero response residual establishes failure of this prediction for the chosen observable and intervention.

Define
$$
\Delta_{OB,\alpha}(t)=\widehat\chi_{OB,\epsilon}(t)
-\beta[C_{OB}(0)-C_{OB}(t)].
$$
Compare its size with the size of the measured response. A small absolute discrepancy can be important when the response itself is small. A relative error is meaningful only when the response scale sits above the numerical error floor.

**Exercise 5.** For each fixture:

1. Run the forward field/readout pair for the values in `ALPHAS`. Reuse your functions from Exercises 1–4. Use the same $\beta,g,h^{(0)},\theta$ and field amplitudes for each $\alpha$. For each $\alpha$, solve for its own resting $\pi_0$.
2. For each condition record $D(P_0)$, the resting mean of $O$, and $\max_t|\Delta_{OB,\alpha}(t)|$. Use `stationary_current` from Notebook 01 for $D$. Also record the peak absolute measured response. If you include a relative error, state the rule you used when the response was close to zero.
3. Plot the measured response and the covariance prediction together for the symmetric and directed endpoints. Show $\Delta(t)$ in a separate panel or figure. Provide a table or a plot of the peak discrepancy across $\alpha$. Label time as elementary updates. Recalculate the directed endpoint with a smaller $\epsilon$ and compare the two residual curves.
4. At $\alpha=0$ and $\alpha=1$, reverse the intervention: apply the same per-cell field to the former readout and average over the former input cells. Rebuild the stimulus mask and both observable vectors. This asks a different physical question than the covariance reciprocity check in Exercise 3. When the two populations have different sizes, swapping them also changes the number of cells receiving the field. Discuss this when comparing the two directions.

**Caution for interpretation.** The interpolation can also change coupling strength and resting activity. This assignment tests exact prediction under a defined family of models. Isolating effects of asymmetry at matched strength or matching the stationary activity will require additional controls on larger circuits.

In [ ]:
# Your code for Exercise 5.
# Measure the response and its residual across alpha and direction.

## 6 · Explain what the comparison establishes

Use your results to write four short paragraphs.

1. **Control.** Which checks establish that the equilibrium comparison works? Explain why a failure at $\alpha=0$ would make a directed comparison hard to interpret.
2. **Directed response.** Where in time did the prediction fail, for which input–readout direction, and by how much? Compare the absolute residual with the response amplitude and with the change caused by halving or otherwise reducing $\epsilon$.
3. **Two diagnostics.** Distinguish what $D(P_0)$ establishes from what $\Delta_{OB,\alpha}$ establishes. Could an irreversible chain produce a response close to the equilibrium predictor for a particular observable?
4. **From the small circuit to the paper.** The three-cell MaleCNS model omits most effective input to each selected cell, even though the denominators use all eligible inputs. Explain what our exact experiment verifies and what remains to be tested using trajectories and broader anatomical coverage. How would you check a trajectory estimator using the exact results here?

**Your answer · Exercise 6**

1. Control
2. Directed response
3. Two diagnostics
4. From the small circuit to the paper

## Submission and next experiment

Restart the kernel and run the completed assignment top to bottom. Include the one-cell checks, the equilibrium field-size sweep, response and residual plots, the direction comparison, and your written interpretation.

The next stage will replace exact propagation with sampled trajectories on these same test cases. The exact curves will serve as references for burn-in, time-lag alignment, replicate uncertainty, and the choice of field increment before scaling up to larger subgraphs.